# Ship Positions Pipeline — Magic Commands Runbook

This notebook runs the pipeline end-to-end using IPython magic commands (`%cd`, `%ls`, `%pip`, `%run`, `%time`) and validates the silver output tables.

In [25]:
from pathlib import Path
import platform

WSL_ROOT = Path("/mnt/c/Users/maorb/work/ship_positions")
WIN_ROOT = Path(r"C:\Users\maorb\work\ship_positions")

PROJECT_ROOT = WSL_ROOT if WSL_ROOT.exists() else WIN_ROOT
print("Kernel OS:", platform.system())
print("Project root resolved to:", PROJECT_ROOT)

%cd $PROJECT_ROOT
%pwd

Kernel OS: Windows
Project root resolved to: C:\Users\maorb\work\ship_positions
C:\Users\maorb\work\ship_positions


'C:\\Users\\maorb\\work\\ship_positions'

In [26]:
import os
os.getcwd()

'C:\\Users\\maorb\\work\\ship_positions'

In [27]:
%ls

print("\nFiles in src/")
for p in sorted((PROJECT_ROOT / "src").iterdir()):
    print(" -", p.name)

print("\nFiles in elt_utils/sql/")
for p in sorted((PROJECT_ROOT / "elt_utils" / "sql").iterdir()):
    print(" -", p.name)

print("\nFiles in data/raw/")
for p in sorted((PROJECT_ROOT / "data" / "raw").iterdir()):
    print(" -", p.name)

 Volume in drive C is OS
 Volume Serial Number is ACB1-0DAC

 Directory of C:\Users\maorb\work\ship_positions

08/03/2026  20:23    <DIR>          .
08/03/2026  10:43    <DIR>          ..
08/03/2026  10:43             6,148 .DS_Store
08/03/2026  13:00    <DIR>          .pytest_cache
08/03/2026  20:21    <DIR>          .venv-1
08/03/2026  10:43    <DIR>          data
08/03/2026  19:47                 0 EDA.ipynb
08/03/2026  11:30    <DIR>          elt_utils
08/03/2026  11:52            15,429 Pipeline_Magic_Runbook.ipynb
08/03/2026  10:51             6,140 prompt.md
08/03/2026  20:27             5,023 README.md
08/03/2026  10:43                50 requirements.txt
08/03/2026  10:43    <DIR>          src
08/03/2026  10:43    <DIR>          tests
08/03/2026  11:50    <DIR>          venvo
               6 File(s)         32,790 bytes
               9 Dir(s)   3,441,299,456 bytes free

Files in src/
 - bl
 - config
 - pipeline
 - queries
 - run_me.py

Files in elt_utils/sql/
 - create_reject

In [28]:
%pip install -r requirements.txt

Note: you may need to restart the kernel to use updated packages.


In [29]:
%pycat src/config/config.yaml

pipeline:
  source_db: data/raw/ship_positions.db
  target_db: data/silver/ship_positions.db
  source_schema: raw
  source_table: ship_positions
  target_schema: silver
  target_table: SHIP_POSITIONS
  rejects_table: SHIP_POSITIONS_REJECTS
  sql_dir: elt_utils/sql

dqa:
  lat_min: -90.0
  lat_max: 90.0
  lon_min: -180.0
  lon_max: 180.0
  sog_min: 0.0
  sog_max: 100.0
  max_sog_diff: 40.0
  cog_min: 0.0
  cog_max: 360.0

queries:
  create_target_table: create_target_table.sql
  create_rejects_table: create_rejects_table.sql
  extract_delta: extract_delta.sql
  fetch_target_lookback: fetch_target_lookback.sql
  insert_rejects: insert_rejects.sql
  insert_target_rows: insert_target_rows.sql


## Optional: Reset target DB for a fresh run

Set `RESET_TARGET = True` if you want to rerun from scratch and reproduce full-load publish counts from an empty silver DB.

In [30]:
TARGET_DB = PROJECT_ROOT / "data" / "silver" / "ship_positions.db"
RESET_TARGET = False

if RESET_TARGET and TARGET_DB.exists():
    TARGET_DB.unlink()
    print("Deleted existing target DB:", TARGET_DB)
else:
    print("Keeping existing target DB:", TARGET_DB)

Keeping existing target DB: C:\Users\maorb\work\ship_positions\data\silver\ship_positions.db


## 1) First Load

In [31]:
%time %run src/run_me.py --source-db data/raw/ship_positions.db

2026-03-08 20:30:14,342 | INFO | __main__ | Starting pipeline
2026-03-08 20:30:14,375 | INFO | pipeline.pipe | Pipeline starting | source_db=C:\Users\maorb\work\ship_positions\data\raw\ship_positions.db | target_db=C:\Users\maorb\work\ship_positions\data\silver\ship_positions.db


2026-03-08 20:30:14,580 | INFO | pipeline.pipe | Target tables initialized
2026-03-08 20:30:15,146 | INFO | pipeline.pipe | Delta extracted | rows=134636
2026-03-08 20:30:15,430 | INFO | pipeline.pipe | DQA complete | clean_rows=132480 | rejected_rows=2156
2026-03-08 20:30:15,451 | INFO | pipeline.pipe | Lookback fetched | rows=0
2026-03-08 20:30:15,933 | INFO | pipeline.pipe | Enrichment complete | rows=132480
2026-03-08 20:30:16,608 | INFO | pipeline.pipe | Publish complete | published_rows=132480 | rejected_rows=2156
2026-03-08 20:30:16,608 | INFO | pipeline.pipe | Pipeline completed successfully
2026-03-08 20:30:16,652 | INFO | __main__ | Pipeline summary: {'source_db': 'C:\\Users\\maorb\\work\\ship_positions\\data\\raw\\ship_positions.db', 'target_db': 'C:\\Users\\maorb\\work\\ship_positions\\data\\silver\\ship_positions.db', 'delta_rows': 134636, 'clean_rows': 132480, 'rejected_rows': 2156, 'published_rows': 132480}


{
  "source_db": "C:\\Users\\maorb\\work\\ship_positions\\data\\raw\\ship_positions.db",
  "target_db": "C:\\Users\\maorb\\work\\ship_positions\\data\\silver\\ship_positions.db",
  "delta_rows": 134636,
  "clean_rows": 132480,
  "rejected_rows": 2156,
  "published_rows": 132480
}
CPU times: total: 3.08 s
Wall time: 2.36 s


## 2) Incremental Load

In [32]:
%time %run src/run_me.py --source-db data/raw/ship_positions_incremental.db

2026-03-08 20:30:17,581 | INFO | __main__ | Starting pipeline
2026-03-08 20:30:17,601 | INFO | pipeline.pipe | Pipeline starting | source_db=C:\Users\maorb\work\ship_positions\data\raw\ship_positions_incremental.db | target_db=C:\Users\maorb\work\ship_positions\data\silver\ship_positions.db
2026-03-08 20:30:17,684 | INFO | pipeline.pipe | Target tables initialized
2026-03-08 20:30:17,767 | INFO | pipeline.pipe | Delta extracted | rows=12554
2026-03-08 20:30:17,822 | INFO | pipeline.pipe | DQA complete | clean_rows=12052 | rejected_rows=502
2026-03-08 20:30:17,889 | INFO | pipeline.pipe | Lookback fetched | rows=2
2026-03-08 20:30:17,975 | INFO | pipeline.pipe | Enrichment complete | rows=12052
2026-03-08 20:30:18,086 | INFO | pipeline.pipe | Publish complete | published_rows=12052 | rejected_rows=502
2026-03-08 20:30:18,086 | INFO | pipeline.pipe | Pipeline completed successfully
2026-03-08 20:30:18,140 | INFO | __main__ | Pipeline summary: {'source_db': 'C:\\Users\\maorb\\work\\ship_p

{
  "source_db": "C:\\Users\\maorb\\work\\ship_positions\\data\\raw\\ship_positions_incremental.db",
  "target_db": "C:\\Users\\maorb\\work\\ship_positions\\data\\silver\\ship_positions.db",
  "delta_rows": 12554,
  "clean_rows": 12052,
  "rejected_rows": 502,
  "published_rows": 12052
}
CPU times: total: 875 ms
Wall time: 571 ms


## 3) Idempotency Check (Run Incremental Again)

In [33]:
%time %run src/run_me.py --source-db data/raw/ship_positions_incremental.db

2026-03-08 20:30:18,168 | INFO | __main__ | Starting pipeline
2026-03-08 20:30:18,187 | INFO | pipeline.pipe | Pipeline starting | source_db=C:\Users\maorb\work\ship_positions\data\raw\ship_positions_incremental.db | target_db=C:\Users\maorb\work\ship_positions\data\silver\ship_positions.db
2026-03-08 20:30:18,236 | INFO | pipeline.pipe | Target tables initialized
2026-03-08 20:30:18,274 | INFO | pipeline.pipe | Delta extracted | rows=502
2026-03-08 20:30:18,309 | INFO | pipeline.pipe | DQA complete | clean_rows=0 | rejected_rows=502
2026-03-08 20:30:18,388 | INFO | pipeline.pipe | Lookback fetched | rows=2
2026-03-08 20:30:18,391 | INFO | pipeline.pipe | Enrichment complete | rows=0
2026-03-08 20:30:18,408 | INFO | pipeline.pipe | Publish complete | published_rows=0 | rejected_rows=502
2026-03-08 20:30:18,412 | INFO | pipeline.pipe | Pipeline completed successfully
2026-03-08 20:30:18,446 | INFO | __main__ | Pipeline summary: {'source_db': 'C:\\Users\\maorb\\work\\ship_positions\\data

{
  "source_db": "C:\\Users\\maorb\\work\\ship_positions\\data\\raw\\ship_positions_incremental.db",
  "target_db": "C:\\Users\\maorb\\work\\ship_positions\\data\\silver\\ship_positions.db",
  "delta_rows": 502,
  "clean_rows": 0,
  "rejected_rows": 502,
  "published_rows": 0
}
CPU times: total: 406 ms
Wall time: 282 ms


## 4) Validate Target Tables

In [36]:
import duckdb

con = duckdb.connect('data/silver/ship_positions.db')
target_rows = con.execute('SELECT COUNT(*) FROM silver.SHIP_POSITIONS').fetchone()[0]
reject_rows = con.execute('SELECT COUNT(*) FROM silver.SHIP_POSITIONS_REJECTS').fetchone()[0]
dup_keys = con.execute('''
SELECT COUNT(*)
FROM (
  SELECT ship_name, time_ts, COUNT(*) AS cnt
  FROM silver.SHIP_POSITIONS
  GROUP BY 1,2
  HAVING COUNT(*) > 1
)
''').fetchone()[0]

print({
    'target_rows': target_rows,
    'reject_rows': reject_rows,
    'duplicate_keys': dup_keys
})

con.close()

{'target_rows': 144532, 'reject_rows': 5316, 'duplicate_keys': 0}


## 5) Print sample rows (actual data preview)

This section fetches actual row samples from the silver target table, the rejects table, and the raw source table.

In [37]:
import duckdb

target_db = PROJECT_ROOT / "data" / "silver" / "ship_positions.db"
raw_full_db = PROJECT_ROOT / "data" / "raw" / "ship_positions.db"
raw_incremental_db = PROJECT_ROOT / "data" / "raw" / "ship_positions_incremental.db"

with duckdb.connect(str(target_db)) as con_target:
    sample_target = con_target.execute(
        """
        SELECT
            ship_name, time_ts, lat, lon, sog, cog, rot, acceleration, distance_traveled, elt_published_at
        FROM silver.SHIP_POSITIONS
        ORDER BY time_ts DESC
        LIMIT 10
        """
    ).df()

    sample_rejects = con_target.execute(
        """
        SELECT
            ship_name, raw_time, lat, lon, sog, cog, dq_reason, elt_rejected_at
        FROM silver.SHIP_POSITIONS_REJECTS
        ORDER BY elt_rejected_at DESC
        LIMIT 10
        """
    ).df()

with duckdb.connect(str(raw_full_db)) as con_raw:
    sample_raw = con_raw.execute(
        """
        SELECT ship_name, time, lat, lon, sog, cog
        FROM raw.ship_positions
        ORDER BY time DESC
        LIMIT 10
        """
    ).df()

with duckdb.connect(str(raw_incremental_db)) as con_incr:
    sample_incremental = con_incr.execute(
        """
        SELECT ship_name, time, lat, lon, sog, cog
        FROM raw.ship_positions
        ORDER BY time DESC
        LIMIT 10
        """
    ).df()

print("Sample rows from silver.SHIP_POSITIONS")
display(sample_target)

print("Sample rows from silver.SHIP_POSITIONS_REJECTS")
display(sample_rejects)

print("Sample rows from raw.ship_positions (full source DB)")
display(sample_raw)

print("Sample rows from raw.ship_positions_incremental (incremental source DB)")
display(sample_incremental)

Sample rows from silver.SHIP_POSITIONS


,ship_name,time_ts,lat,lon,sog,cog,rot,acceleration,distance_traveled,elt_published_at
0,Ship_A,2026-02-24 07:59:48.511,30.544876,122.213852,1.122668,99.377472,-1.829765,-0.029094,0.009584,2026-03-08 18:30:17.976891
1,Ship_B,2026-02-24 07:59:47.678,1.359049,-13.351282,9.648289,244.004654,1.258698,0.220932,0.080254,2026-03-08 18:30:17.976891
2,Ship_A,2026-02-24 07:59:18.511,30.544901,122.213669,1.151762,101.207237,0.495743,0.008332,0.009257,2026-03-08 18:30:17.976891
3,Ship_B,2026-02-24 07:59:17.678,1.359635,-13.350080,9.427357,242.745956,0.230515,NaN,0.078490,2026-03-08 18:30:17.976891
4,Ship_A,2026-02-24 07:58:48.511,30.544931,122.213493,1.143431,100.711494,-0.759949,-0.013138,0.009644,2026-03-08 18:30:17.976891
5,Ship_B,2026-02-24 07:58:47.678,1.360234,-13.348918,NaN,242.515442,-0.776031,NaN,0.079688,2026-03-08 18:30:17.976891
6,Ship_A,2026-02-24 07:58:18.511,30.544962,122.213310,1.156568,101.471443,-2.461243,0.000533,0.009644,2026-03-08 18:30:17.976891
7,Ship_B,2026-02-24 07:58:17.678,1.360847,-13.347740,9.720613,243.291473,1.066376,0.188880,0.080929,2026-03-08 18:30:17.976891
8,Ship_A,2026-02-24 07:57:48.511,30.544992,122.213127,1.156036,103.932686,-5.256302,0.052602,0.009769,2026-03-08 18:30:17.976891
9,Ship_B,2026-02-24 07:57:47.678,1.361452,-13.346536,9.531734,242.225098,1.668152,-0.013797,0.079321,2026-03-08 18:30:17.976891


Sample rows from silver.SHIP_POSITIONS_REJECTS


,ship_name,raw_time,lat,lon,sog,cog,dq_reason,elt_rejected_at
0,Ship_A,1.769600e+09,0.0,0.0,18.299999,60.400002,gps_zero_coordinates,2026-03-08 18:33:35.367466
1,Ship_A,1.769600e+09,0.0,0.0,18.400000,60.599998,gps_zero_coordinates,2026-03-08 18:33:35.367466
2,Ship_A,1.769602e+09,0.0,0.0,18.000000,64.400002,gps_zero_coordinates,2026-03-08 18:33:35.367466
3,Ship_A,1.769604e+09,0.0,0.0,18.000000,64.199997,gps_zero_coordinates,2026-03-08 18:33:35.367466
4,Ship_A,1.769604e+09,0.0,0.0,18.000000,63.799999,gps_zero_coordinates,2026-03-08 18:33:35.367466
5,Ship_A,1.769605e+09,0.0,0.0,17.900000,61.799999,gps_zero_coordinates,2026-03-08 18:33:35.367466
6,Ship_A,1.769607e+09,0.0,0.0,17.799999,61.799999,gps_zero_coordinates,2026-03-08 18:33:35.367466
7,Ship_A,1.769609e+09,0.0,0.0,18.000000,61.700001,gps_zero_coordinates,2026-03-08 18:33:35.367466
8,Ship_A,1.769612e+09,0.0,0.0,17.500000,62.400002,gps_zero_coordinates,2026-03-08 18:33:35.367466
9,Ship_A,1.769625e+09,0.0,0.0,16.500000,59.500000,gps_zero_coordinates,2026-03-08 18:33:35.367466


Sample rows from raw.ship_positions (full source DB)


,ship_name,time,lat,lon,sog,cog
0,Ship_B,1.771732e+09,3.738817,-12.279050,14.7,144.0
1,Ship_B,1.771732e+09,3.740517,-12.280233,14.8,144.0
2,Ship_B,1.771732e+09,3.742200,-12.281450,14.7,144.0
3,Ship_B,1.771732e+09,3.743817,-12.282600,14.8,143.0
4,Ship_B,1.771732e+09,3.745600,-12.283883,14.8,143.0
5,Ship_B,1.771732e+09,3.747283,-12.285084,14.8,143.0
6,Ship_B,1.771732e+09,3.748917,-12.286250,14.8,143.0
7,Ship_B,1.771732e+09,3.750600,-12.287467,14.9,143.0
8,Ship_B,1.771732e+09,3.752283,-12.288716,14.9,143.0
9,Ship_B,1.771732e+09,34.861485,-39.430996,14.9,144.0


Sample rows from raw.ship_positions_incremental (incremental source DB)


,ship_name,time,lat,lon,sog,cog
0,Ship_A,1.771920e+09,30.544876,122.213852,1.122668,99.377472
1,Ship_B,1.771920e+09,1.359049,-13.351282,9.648289,244.004654
2,Ship_A,1.771920e+09,30.544901,122.213669,1.151762,101.207237
3,Ship_B,1.771920e+09,1.359635,-13.350080,9.427357,242.745956
4,Ship_A,1.771920e+09,30.544931,122.213493,1.143431,100.711494
5,Ship_B,1.771920e+09,1.360234,-13.348918,-5.810600,242.515442
6,Ship_A,1.771920e+09,30.544962,122.213310,1.156568,101.471443
7,Ship_B,1.771920e+09,1.360847,-13.347740,9.720613,243.291473
8,Ship_A,1.771920e+09,30.544992,122.213127,1.156036,103.932686
9,Ship_B,1.771920e+09,1.361452,-13.346536,9.531734,242.225098
